Dataset Overview: DAIC-WOZ
**Pipeline v22** — Klasifikasi Kesehatan Mental Berbasis Audio

─────────────────────────────────────────────────────────────────────
 v22 = LOOCV + ADASYN + PCA + Ensemble

 [1] 3 Ekstraksi Fitur (MFCC, Spectrogram, Wav2Vec 2.0 Full)
 [2] Menggunakan keseluruhan data (102 partisipan) dengan LOOCV
 [3] Terdapat 4 model dasar: LR, SVM, XGBoost, Random Forest
 [4] Pipeline preprocessing di dalam LOOCV:
     Imputasi -> Scaling -> ADASYN -> PCA -> Classifier
 [5] Menambahkan Soft Voting Ensemble di akhir
─────────────────────────────────────────────────────────────────────

## Setup

In [1]:
import subprocess, sys, os, pickle, json, time, warnings
warnings.filterwarnings('ignore')

def _pip(pkg):
    try: __import__(pkg.split('==')[0].split('>=')[0]); return
    except: pass
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_pip("imbalanced-learn")

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report
)
from imblearn.over_sampling import ADASYN, SMOTE
import xgboost as xgb

plt.rcParams['font.family'] = 'DejaVu Sans'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if "notebooks" in os.getcwd() else os.getcwd()
V6_FEAT_DIR = os.path.join(PROJECT_ROOT, "data", "features", "v6")
V8_FEAT_DIR = os.path.join(PROJECT_ROOT, "data", "features", "v8")
MODELS_DIR  = os.path.join(PROJECT_ROOT, "models", "ml_v22")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v22")

for d in [MODELS_DIR, os.path.join(RESULTS_DIR, "metrics"),
          os.path.join(RESULTS_DIR, "plots"), os.path.join(RESULTS_DIR, "confusion_matrix")]:
    os.makedirs(d, exist_ok=True)

## Load Features

In [2]:
META_COLS = ['participant_id', 'phq8_score', 'label_depresi', 'gender', 'label']

def load_clean(csv_path, name):
    df = pd.read_csv(csv_path)
    feat_cols = [c for c in df.columns if c not in META_COLS]
    df[feat_cols] = df[feat_cols].fillna(0)
    if 'label' not in df.columns and 'label_depresi' in df.columns:
        df['label'] = df['label_depresi']
    return df, feat_cols

df_mfcc, cols_mfcc = load_clean(os.path.join(V6_FEAT_DIR, "daic_v6_mfcc.csv"), "MFCC")
df_spec, cols_spec = load_clean(os.path.join(V6_FEAT_DIR, "daic_v6_spectrogram.csv"), "Spectrogram")
W2V_FULL_CSV = os.path.join(V8_FEAT_DIR, "daic_v8_wav2vec_full.csv")
df_w2v, cols_w2v = load_clean(W2V_FULL_CSV, "Wav2Vec_Full")

# No early fusion here to keep things fast and stick to ensemble late fusion.
datasets = {
    'MFCC': (df_mfcc, cols_mfcc),
    'Spectrogram': (df_spec, cols_spec),
    'Wav2Vec_Full': (df_w2v, cols_w2v)
}

## Model Config

In [3]:
def get_models():
    return {
        'Logistic Regression': LogisticRegression(
            max_iter=10000, random_state=RANDOM_SEED, class_weight='balanced', C=0.01, solver='liblinear'
        ),
        'SVM': SVC(
            kernel='rbf', probability=True, C=1.0, gamma='scale',
            random_state=RANDOM_SEED, class_weight='balanced'
        ),
        'XGBoost': xgb.XGBClassifier(
            random_state=RANDOM_SEED, eval_metric='logloss',
            objective='binary:logistic', n_jobs=-1,
            scale_pos_weight=2.5, n_estimators=150, max_depth=3,
            learning_rate=0.05, subsample=0.8, colsample_bytree=0.8
        ),
        'Random Forest': RandomForestClassifier(
            random_state=RANDOM_SEED, class_weight='balanced', n_jobs=-1,
            n_estimators=300, max_depth=5, max_features='sqrt'
        ),
    }

MODEL_NAMES = list(get_models().keys())
FEAT_NAMES  = list(datasets.keys())

## LOOCV Evaluation Loop

In [4]:
def loocv_evaluate(df, feat_cols, model_fn):
    n = len(df)
    X = df[feat_cols].values.astype(np.float64)
    y = df['label'].values.astype(int)

    y_true_all, y_pred_all, y_prob_all = np.zeros(n, dtype=int), np.zeros(n, dtype=int), np.zeros(n, dtype=float)

    for i in range(n):
        X_tr, y_tr = np.delete(X, i, axis=0), np.delete(y, i, axis=0)
        X_te = X[i:i+1]

        # NaN handling
        medians = np.nanmedian(X_tr, axis=0)
        np.copyto(X_tr, medians, where=np.isnan(X_tr))
        np.copyto(X_te, medians, where=np.isnan(X_te))

        # Scaling
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)
        
        # ADASYN
        try:
            ada = ADASYN(random_state=RANDOM_SEED, n_neighbors=3)
            X_tr_res, y_tr_res = ada.fit_resample(X_tr, y_tr)
        except:
            sm = SMOTE(random_state=RANDOM_SEED, k_neighbors=3)
            X_tr_res, y_tr_res = sm.fit_resample(X_tr, y_tr)
        
        # PCA (n_components=30)
        k_feats = min(30, X_tr_res.shape[1])
        pca = PCA(n_components=k_feats, random_state=RANDOM_SEED)
        X_tr_res = pca.fit_transform(X_tr_res)
        X_te = pca.transform(X_te)

        # Train & Predict
        model = model_fn()
        model.fit(X_tr_res, y_tr_res)
        
        try: prob = model.predict_proba(X_te)[0, 1]
        except: prob = float(model.predict(X_te)[0])
        
        y_true_all[i] = y[i]
        y_prob_all[i] = prob
        y_pred_all[i] = int(prob >= 0.5)

    metrics = {
        'f1_macro': float(f1_score(y_true_all, y_pred_all, average='macro', zero_division=0)),
        'accuracy': float(accuracy_score(y_true_all, y_pred_all)),
        'roc_auc': float(roc_auc_score(y_true_all, y_prob_all))
    }

    best_thr, best_f1 = 0.5, metrics['f1_macro']
    for thr in np.arange(0.30, 0.71, 0.01):
        preds_t = (y_prob_all >= thr).astype(int)
        f1_t = f1_score(y_true_all, preds_t, average='macro', zero_division=0)
        if f1_t > best_f1: best_f1, best_thr = f1_t, thr

    metrics['f1_tuned'] = float(best_f1)
    metrics['best_threshold'] = float(round(best_thr, 2))
    metrics['acc_tuned'] = float(accuracy_score(y_true_all, (y_prob_all >= best_thr).astype(int)))

    return metrics, y_true_all, y_prob_all

## Running

In [5]:
all_results, all_ys = {}, {}
for feat_name in FEAT_NAMES:
    df, feat_cols = datasets[feat_name]
    print(f"\n[{feat_name}] ({len(feat_cols)} features)")
    for model_name in MODEL_NAMES:
        combo = f"{feat_name} + {model_name}"
        t0 = time.time()
        model_fn = lambda mn=model_name: get_models()[mn]
        metrics, y_true, y_prob = loocv_evaluate(df, feat_cols, model_fn)
        all_results[combo] = metrics
        all_ys[combo] = (y_true, y_prob, metrics['best_threshold'])
        print(f"  {model_name:<20}: F1_tuned={metrics['f1_tuned']:.4f} (thr={metrics['best_threshold']:.2f}) | Time={time.time()-t0:.1f}s")


[MFCC] (990 features)


  Logistic Regression : F1_tuned=0.5146 (thr=0.62) | Time=12.2s


  SVM                 : F1_tuned=0.6178 (thr=0.47) | Time=5.3s


  XGBoost             : F1_tuned=0.5476 (thr=0.57) | Time=7.0s


  Random Forest       : F1_tuned=0.5123 (thr=0.50) | Time=32.6s

[Spectrogram] (722 features)


...

  Random Forest       : F1_tuned=0.5098 (thr=0.41) | Time=32.2s

[Wav2Vec_Full] (1536 features)


  Logistic Regression : F1_tuned=0.5577 (thr=0.48) | Time=6.1s


  SVM                 : F1_tuned=0.5147 (thr=0.54) | Time=5.5s


  XGBoost             : F1_tuned=0.5671 (thr=0.48) | Time=7.4s


  Random Forest       : F1_tuned=0.5827 (thr=0.59) | Time=33.0s


## Ensemble (Soft Voting Top-3)

In [6]:
sorted_combos = sorted(all_results.keys(), key=lambda k: all_results[k]['f1_tuned'], reverse=True)
top3 = sorted_combos[:3]
print(f"\n[Ensemble Top-3]")
for c in top3:
    print(f"  {c}: {all_results[c]['f1_tuned']:.4f}")

y_true_ens = all_ys[top3[0]][0]
probs_top3 = np.array([all_ys[c][1] for c in top3])
y_prob_ens = probs_top3.mean(axis=0)

best_thr_ens, best_f1_ens = 0.5, f1_score(y_true_ens, (y_prob_ens >= 0.5).astype(int), average='macro', zero_division=0)
for thr in np.arange(0.30, 0.71, 0.01):
    preds = (y_prob_ens >= thr).astype(int)
    f1_t = f1_score(y_true_ens, preds, average='macro', zero_division=0)
    if f1_t > best_f1_ens:
        best_f1_ens, best_thr_ens = f1_t, thr

all_results['Ensemble_Top3'] = {
    'f1_tuned': best_f1_ens, 'best_threshold': best_thr_ens,
    'roc_auc': roc_auc_score(y_true_ens, y_prob_ens)
}


[Ensemble Top-3]
  MFCC + SVM: 0.6178
  Wav2Vec_Full + Random Forest: 0.5827
  Wav2Vec_Full + XGBoost: 0.5671


## Save Results

In [7]:
rows = []
for combo, m in all_results.items():
    if combo == 'Ensemble_Top3':
        parts = ['Ensemble', 'Top3']
    else:
        parts = combo.split(' + ')
    rows.append({
        'Feature': parts[0], 'Model': parts[1],
        'F1 (tuned)': m['f1_tuned'], 'Best Thr': m['best_threshold'],
        'AUC': m['roc_auc']
    })

df_results = pd.DataFrame(rows).sort_values('F1 (tuned)', ascending=False).reset_index(drop=True)
df_results.index += 1

csv_path = os.path.join(RESULTS_DIR, "metrics", "v22_results.csv")
df_results.to_csv(csv_path, index=False)

print("\n" + "=" * 80)
print(f"RINGKASAN v22 — LOOCV (102 folds, ADASYN + PCA + Ensemble)")
print("=" * 80)
print(df_results.to_string())
print("\nDone.")


RINGKASAN v22 — LOOCV (102 folds, ADASYN + PCA + Ensemble)
         Feature                Model  F1 (tuned)  Best Thr       AUC
1           MFCC                  SVM    0.617823      0.47  0.536834
2   Wav2Vec_Full        Random Forest    0.582700      0.59  0.582011
3       Ensemble                 Top3    0.567752      0.56  0.576720
4   Wav2Vec_Full              XGBoost    0.567130      0.48  0.564917
5   Wav2Vec_Full  Logistic Regression    0.557745      0.48  0.518926
6           MFCC              XGBoost    0.547581      0.57  0.524217
7    Spectrogram              XGBoost    0.538817      0.38  0.496947
8    Spectrogram                  SVM    0.522807      0.47  0.494912
9    Spectrogram  Logistic Regression    0.517540      0.48  0.463166
10  Wav2Vec_Full                  SVM    0.514706      0.54  0.448107
11          MFCC  Logistic Regression    0.514555      0.62  0.448107
12          MFCC        Random Forest    0.512266      0.50  0.503053
13   Spectrogram        Random